# Generalization Test — fine-tuned models on TabFact (zero TabFact training)

Loads the Condition B and C checkpoints and evaluates them on TabFact. Strong TabFact accuracy with no TabFact training = real transfer, not memorization. Ends with the aggregated summary table + plots.

In [ ]:
# --- Setup: on Colab this clones the repo and installs deps; locally it is a no-op ---
import os
if not os.path.isdir("src"):
    if not os.path.isdir("ECS111FinalProject"):
        !git clone https://github.com/adiseshvsanklapur/ECS111FinalProject.git
    os.chdir("ECS111FinalProject")
    !pip -q install -r requirements.txt
print("cwd:", os.getcwd())

In [ ]:
# Set SMOKE = False for the full, reported run. SMOKE = True does a fast real
# end-to-end pass (flan-t5-small, tiny slice) to confirm everything works first.
SMOKE = True

In [ ]:
from src import config
device = config.get_device()
print("device:", device)

if SMOKE:
    prompt_models = [config.SMOKE_MODEL]
    seeds = [13]
    eval_n = config.SMOKE_EVAL_N
    eval_n_cot = config.SMOKE_EVAL_N
    train_n = config.SMOKE_TRAIN_N
else:
    prompt_models = config.PROMPT_MODELS      # flan-t5-base + large
    seeds = config.SEEDS                       # [13, 42]
    eval_n = config.EVAL_N                      # 1000
    eval_n_cot = config.EVAL_N_COT             # 500
    train_n = config.TRAIN_N                    # 8000

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from src.data import load_tabfact
from src.prompts import build_baseline_prompt
from src.evaluate import predict_and_evaluate

tabfact = load_tabfact(n=eval_n, seed=config.EVAL_SEED)
ft_label = config.SMOKE_MODEL if SMOKE else config.FINETUNE_MODEL

rows = []
for cond in ["finetune_answers", "finetune_traces"]:
    for seed in seeds:
        ckpt = f"checkpoints/{cond}_seed{seed}"
        model = AutoModelForSeq2SeqLM.from_pretrained(ckpt).to(device)
        tok = AutoTokenizer.from_pretrained(ckpt)
        res = predict_and_evaluate(
            model, tok, tabfact, build_baseline_prompt,
            condition=f"generalization_{cond}", model_id=ft_label, seed=seed,
            task="tabfact", device=device,
        )
        print(cond, "seed", seed, res["metrics"])
        rows.append((cond, seed, res["metrics"]))
rows

## Aggregate everything

Run after all condition notebooks have written their results JSONs to `results/`.

In [ ]:
from src import analysis
results = analysis.load_results()
df = analysis.aggregate(results)
display(df)
paths = analysis.write_summary_table(df)
analysis.plot_primary_metric(df, config.RESULTS_DIR / "primary_metric.png")
import glob
wtq_results = [r for r in results if r["task"] == "wtq"]
if wtq_results:
    analysis.plot_error_distribution(results, config.RESULTS_DIR / "error_distribution.png")
print("wrote:", paths)